# 🎬 Tenglish-LM: YouTube Movie Trailer & Scene Extractor (Colab Quick-Test)
**Runtime:** Google Colab **Free CPU** (Switch to T4 GPU *only* if testing Whisper audio transcription on uncaptioned trailers)

### How to Use This Notebook (3 Clicks):
1. **Run Cell 1:** Installs `yt-dlp` and connects your Google Drive (`/MyDrive/Tenglish_LM/trailer_tests`).
2. **Run Cell 2:** Loads the **Tenglish-Evo Engine** (Universal Akshara-to-Latin Transducer, Protected Emotional Core `kopam`/`nijam`/`dhairyam`, Stem-final `-m` fix `yuddhamlo`/`varshamlo`, and Auto-Emotion Tagger).
3. **Run Cell 3:** Paste **any YouTube Telugu Movie Trailer (or Scene) URL** into `YOUTUBE_URL` and run! It outputs:
   * `<trailer_name>_scenes.csv` (opens directly in Google Sheets / Excel)
   * `<trailer_name>_scenes.txt` (human-readable tagged script with all 4 Tenglish Personas)
   * An interactive Pandas table right inside Colab!


In [ ]:
# @title Step 1: Install yt-dlp & Mount Google Drive
!pip install -q -U yt-dlp pandas

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/Tenglish_LM/trailer_tests')
    print("✅ Mounted Google Drive! Files will be saved to:", OUT_DIR)
except Exception:
    OUT_DIR = Path('/content/Tenglish_LM/trailer_tests')
    print("ℹ️ Running in local/ephemeral storage:", OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# @title Step 2: Load the Self-Contained Tenglish-Evo & Scene Emotion Engine
# Contains: Universal Akshara-to-Latin Transducer + Protected Emotional Core + SRT Pause-Gap Slicer

"""
Milestone 2: Tenglish-Evo Engine
(Universal Telugu Akshara-to-Latin Transducer + Protected Emotional Core + Natural Movie English Policy)

Incorporates Gate 1-2 Human-in-the-Loop Audit Fixes:
1. Stem-Final Anusvara ('ం') Preservation before Case Suffixes:
   - 'యుద్ధంలో' -> 'yuddhamlo' / 'yudhamlo' (NEVER 'yuddanlo')
   - 'వర్షంలో' -> 'varshamlo' (NEVER 'varshanlo')
   - 'జీవితంలో' -> 'jeevithamlo' / 'jivitamlo' (NEVER 'jeevithanlo')
2. Natural Conjunct Digraph Smoothing:
   - 'ద్ధ' -> 'ddh' (P1) / 'dh' (P2) ('yuddhamlo', 'siddhu' / 'siddu', not 'yudhdhanlo' or 'sidhdhoo')
   - 'త్త' -> 'tth' (P1) / 'th' (P2) ('mottham' / 'motham', not 'moththam')
   - 'చ్చ' -> 'ch' ('vachanu', 'ichav', 'nachindi', not 'ichchaav')
3. Protected Telugu Emotional Core vs. Natural Movie English Policy:
   - Core emotional & philosophical Telugu words ('kopam', 'nijam', 'dhairyam', 'prema', 'baadha',
     'bhayam', 'pranam', 'manasu', 'kashtam', 'abaddham') are PROTECTED and NEVER swapped to English.
   - Institutional, urban, and light-verb terms naturally spoken in English in Telugu movies
     ('life', 'career', 'decision', 'settle', 'coffee', 'office', 'engagement', 'happy-ga') stay in English.
"""

import math
import re
from collections import Counter
from typing import Any, Dict, List, Set, Tuple


# ---------------------------------------------------------------------------
# 1. Protected Telugu Emotional Core (NEVER overwrite with English!)
# ---------------------------------------------------------------------------
PROTECTED_TELUGU_EMOTIONAL_CORE: Set[str] = {
    "కోపం",       # kopam (anger - never swap to 'angry'/'trigger')
    "నిజం",       # nijam (truth - never swap to 'fact')
    "ధైర్యం",      # dhairyam (courage - never swap to 'guts')
    "ప్రేమ",       # prema (love)
    "బాధ",        # baadha (pain/grief)
    "భయం",        # bhayam (fear)
    "ప్రాణం",      # pranam (life-force/soul)
    "మనసు",       # manasu (heart/mind)
    "కష్టం",       # kashtam (hardship)
    "అబద్ధం",      # abaddham (lie)
    "అబద్ధమేనా",   # abaddhamena
    "మోసం",       # mosam (betrayal)
    "స్నేహం",      # sneham (friendship)
    "యుద్ధం",      # yuddham (war/battle)
    "యుద్ధంలో",    # yuddhamlo (in battle)
    "శత్రువుని",   # shathruvuni (enemy)
    "గెలవడం",     # gelavadam (winning)
    "ఓడిపోయినా",   # odipoyina (even if defeated)
}


# ---------------------------------------------------------------------------
# 2. Universal Telugu Unicode (0x0C00-0x0C7F) Phonetic Tables
# ---------------------------------------------------------------------------

INDEPENDENT_VOWELS_P1 = {
    "అ": "a", "ఆ": "aa", "ఇ": "i", "ఈ": "ee", "ఉ": "u", "ఊ": "oo",
    "ఋ": "ru", "ౠ": "roo", "ఎ": "e", "ఏ": "ye", "ఐ": "ai",
    "ఒ": "o", "ఓ": "o", "ఔ": "au",
}

INDEPENDENT_VOWELS_P2 = {
    "అ": "a", "ఆ": "a", "ఇ": "i", "ఈ": "i", "ఉ": "u", "ఊ": "u",
    "ఋ": "ru", "ౠ": "ru", "ఎ": "e", "ఏ": "e", "ఐ": "ai",
    "ఒ": "o", "ఓ": "o", "ఔ": "au",
}

CONSONANTS_P1 = {
    "క": "k", "ఖ": "kh", "గ": "g", "ఘ": "gh", "ఙ": "ng",
    "చ": "ch", "ఛ": "chh", "జ": "j", "ఝ": "jh", "ఞ": "ny",
    "ట": "t", "ఠ": "th", "డ": "d", "ఢ": "dh", "ణ": "n",
    "త": "th", "థ": "th", "ద": "dh", "ధ": "dh", "న": "n",
    "ప": "p", "ఫ": "ph", "బ": "b", "భ": "bh", "మ": "m",
    "య": "y", "ర": "r", "ఱ": "r", "ల": "l", "ళ": "l",
    "వ": "v", "శ": "sh", "ష": "sh", "స": "s", "హ": "h",
}

CONSONANTS_P2 = {
    "క": "k", "ఖ": "k", "గ": "g", "ఘ": "g", "ఙ": "ng",
    "చ": "ch", "ఛ": "ch", "జ": "j", "ఝ": "j", "ఞ": "ny",
    "ట": "t", "ఠ": "t", "డ": "d", "ఢ": "d", "ణ": "n",
    "త": "t", "థ": "t", "ద": "d", "ధ": "d", "న": "n",
    "ప": "p", "ఫ": "f", "బ": "b", "భ": "b", "మ": "m",
    "య": "y", "ర": "r", "ఱ": "r", "ల": "l", "ళ": "l",
    "వ": "v", "శ": "s", "ష": "sh", "స": "s", "హ": "h",
}

VOWEL_MATRAS_P1 = {
    "ా": "aa", "ి": "i", "ీ": "ee", "ు": "u", "ూ": "u",
    "ృ": "ru", "ె": "e", "ే": "e", "ై": "ai", "ొ": "o", "ో": "o", "ౌ": "au",
}

VOWEL_MATRAS_P2 = {
    "ా": "a", "ి": "i", "ీ": "i", "ు": "u", "ూ": "u",
    "ృ": "ru", "ె": "e", "ే": "e", "ై": "ai", "ొ": "o", "ో": "o", "ౌ": "ow",
}

VIRAMA = "్"
# Sunna ('ం') stays 'm' at word end, before labials (ప,ఫ,బ,భ,మ,వ), AND before case suffixes/liquids (ల,ర,య,స,శ,ష,హ)
# e.g., యుద్ధంలో -> yuddhamlo, వర్షంలో -> varshamlo, జీవితంలో -> jeevithamlo, నిజంగా -> nijamga / nijanga
SUNNA_M_FOLLOWERS = {"ప", "ఫ", "బ", "భ", "మ", "వ", "ల", "ళ", "ర", "య", "శ", "ష", "స", "హ"}


def transliterate_telugu_word_universal(word: str, persona: str = "P1_STANDARD") -> str:
    """
    Universal character/akshara state-machine transducer that converts ANY Telugu Unicode word
    into natural Romanized Tenglish according to the chosen Texter Persona.
    Correctly preserves stem-final '-m' before case suffixes ('yuddhamlo', 'varshamlo', 'jeevithamlo')
    and smooths conjunct digraphs ('yuddham', 'motham', 'siddu', 'ichav').
    """
    use_minimal = persona in {"P2_MINIMAL", "P4_ACROPHON"}
    ivowels = INDEPENDENT_VOWELS_P2 if use_minimal else INDEPENDENT_VOWELS_P1
    cons_map = CONSONANTS_P2 if use_minimal else CONSONANTS_P1
    matras = VOWEL_MATRAS_P2 if use_minimal else VOWEL_MATRAS_P1

    chars = list(word)
    n = len(chars)
    out: List[str] = []
    i = 0

    while i < n:
        ch = chars[i]

        if ch in {"\u200c", "\u200d"}:
            i += 1
            continue

        # 1. Independent Vowel
        if ch in ivowels:
            out.append(ivowels[ch])
            i += 1
            continue

        # 2. Consonant (possibly followed by virama or vowel matra)
        if ch in cons_map:
            c_rom = cons_map[ch]
            next_ch = chars[i + 1] if i + 1 < n else ""

            if next_ch == VIRAMA:
                out.append(c_rom)
                i += 2
                continue
            elif next_ch in matras:
                v_rom = matras[next_ch]
                if next_ch == "ా" and (i + 2 == n) and persona != "P3_EXPRESS":
                    v_rom = "a"
                elif next_ch == "ే" and persona == "P3_EXPRESS" and (i + 2 == n):
                    v_rom = "ey"
                out.append(c_rom + v_rom)
                i += 2
                continue
            else:
                out.append(c_rom + "a")
                i += 1
                continue

        # 3. Nasal Anusvara / Sunna ('ం')
        # Preserves 'm' at word end and before case suffixes like -lo, -la, -va, -sa ('yuddhamlo', 'varshamlo')
        if ch == "ం":
            next_ch = chars[i + 1] if i + 1 < n else ""
            if next_ch == "" or next_ch in SUNNA_M_FOLLOWERS:
                out.append("m")
            else:
                out.append("n")
            i += 1
            continue

        # 4. Visarga / Ardhavisarga
        if ch in {"ః", "ఁ"}:
            out.append("h" if ch == "ః" else "n")
            i += 1
            continue

        # 5. Standalone matra
        if ch in matras:
            out.append(matras[ch])
            i += 1
            continue

        out.append(ch)
        i += 1

    roman = "".join(out)

    # Smooth natural Tenglish conjunct digraphs (prevent ugly 'dhdh', 'thth', 'chch')
    roman = (
        roman.replace("dhdh", "ddh" if not use_minimal else "dh")
        .replace("thth", "tth" if not use_minimal else "th")
        .replace("chch", "ch")
        .replace("shsh", "sh")
    )

    if persona == "P2_MINIMAL":
        for dbl in ("kk", "pp", "ll", "nn", "tt", "dd", "mm", "vv"):
            roman = roman.replace(dbl, dbl[0])
    elif persona == "P3_EXPRESS":
        if roman.endswith(("a", "i", "u", "e", "o")) and len(roman) > 3:
            roman = roman + roman[-1]
    elif persona == "P4_ACROPHON":
        roman = (
            roman.replace("chep", "chap")
            .replace("telu", "talu")
            .replace("vell", "vall")
            .replace("pett", "patt")
        )
        for dbl in ("kk", "pp", "nn", "tt"):
            roman = roman.replace(dbl, dbl[0])

    return roman


# ---------------------------------------------------------------------------
# 3. Natural Movie English Lexicon & Telugu-Script Loanwords
# ---------------------------------------------------------------------------

ENGLISH_CORE_WORDS = {
    "bro", "boss", "sir", "madam", "college", "bus", "stop", "lecturer", "permission",
    "police", "station", "cool", "coffee", "feeling", "operation", "commit", "warning",
    "life", "decision", "career", "settle", "decide", "feelings", "value", "plan",
    "calculation", "normal", "next", "move", "guess", "exactly", "emotion", "pure",
    "logic", "game", "job", "business", "start", "first", "swimming", "direct", "lite",
    "reject", "office", "room", "connect", "sudden", "option", "party", "situation",
    "control", "full", "trauma", "scene", "create", "damage", "message", "seen",
    "reply", "english", "ghost", "delulu", "vibe", "hint", "weekend", "trip", "buildup",
    "last", "minute", "hand", "manager", "friday", "evening", "client", "call", "mind",
    "block", "better", "matter", "update", "point", "topic", "status", "story", "level",
    "attitude", "focus", "serious", "actual", "basic", "cringe", "toxic", "chill",
    "hero", "villain", "interval", "climax", "project", "meeting", "deadline", "salary",
    "girlfriend", "breakup", "patchup", "friend", "friends", "phone", "mobile", "car",
    "bike", "traffic", "signal", "hotel", "bill", "table", "order", "waiter", "doctor",
    "hospital", "ticket", "cinema", "theater", "popcorn", "love", "heart", "brain",
    "shirt", "shoe", "size", "construction", "company", "bank", "loan", "sanction",
    "talent", "engagement", "fix", "already", "ice", "cream", "kick", "secret", "time",
    "happy", "happiness", "long", "uncle", "aunty", "family", "silent", "fine", "please",
    "set", "rest", "kitchen", "sorry", "help", "mess", "jam", "class", "guarantee",
}

TELUGU_SCRIPT_ENGLISH_LOANS = {
    "కాలేజీ": ("college", "EN"),
    "కాలేజీకి": ("college-ki", "HYBRID_FUSED"),
    "బస్": ("bus", "EN"),
    "స్టాప్": ("stop", "EN"),
    "లెక్చరర్": ("lecturer", "EN"),
    "పర్మిషన్": ("permission", "EN"),
    "పోలీస్": ("police", "EN"),
    "స్టేషన్": ("station", "EN"),
    "కూల్": ("cool", "EN"),
    "కాఫీ": ("coffee", "EN"),
    "ఫీలింగ్": ("feeling", "EN"),
    "ఆపరేషన్": ("operation", "EN"),
    "కమిట్": ("commit", "LIGHT_VERB"),
    "వార్నింగ్": ("warning", "EN"),
    "లైఫ్": ("life", "EN"),
    "లైఫ్‌లో": ("life-lo", "HYBRID_FUSED"),
    "డెసిషన్": ("decision", "EN"),
    "కెరీర్": ("career", "EN"),
    "సెటిల్": ("settle", "LIGHT_VERB"),
    "డిసైడ్": ("decide", "LIGHT_VERB"),
    "ఫీలింగ్స్": ("feelings", "EN"),
    "వాల్యూ": ("value", "EN"),
    "ప్లాన్": ("plan", "EN"),
    "క్యాలిక్యులేషన్": ("calculation", "EN"),
    "నార్మల్": ("normal", "EN"),
    "నెక్స్ట్": ("next", "EN"),
    "మూవ్": ("move", "EN"),
    "గెస్": ("guess", "LIGHT_VERB"),
    "ఎగ్జాక్ట్‌లీ": ("exactly", "EN"),
    "ఎమోషన్": ("emotion", "EN"),
    "ప్యూర్": ("pure", "EN"),
    "లాజిక్": ("logic", "EN"),
    "గేమ్": ("game", "EN"),
    "జాబ్": ("job", "EN"),
    "బిజినెస్": ("business", "EN"),
    "స్టార్ట్": ("start", "LIGHT_VERB"),
    "ఫస్ట్": ("first", "EN"),
    "స్విమ్మింగ్": ("swimming", "EN"),
    "డైరెక్ట్": ("direct", "EN"),
    "లైట్": ("lite", "EN"),
    "రిజెక్ట్": ("reject", "LIGHT_VERB"),
    "ఆఫీస్": ("office", "EN"),
    "ఆఫీస్‌కి": ("office-ki", "HYBRID_FUSED"),
    "రూమ్": ("room", "EN"),
    "కనెక్ట్": ("connect", "LIGHT_VERB"),
    "సడన్": ("sudden", "EN"),
    "ఆప్షన్": ("option", "EN"),
    "బ్రో": ("bro", "EN"),
    "పార్టీకి": ("party-ki", "HYBRID_FUSED"),
    "సిట్యుయేషన్": ("situation", "EN"),
    "కంట్రోల్": ("control", "EN"),
    "ఫుల్": ("full", "EN"),
    "ట్రామా": ("trauma", "EN"),
    "సీన్": ("scene", "EN"),
    "క్రియేట్": ("create", "LIGHT_VERB"),
    "డ్యామేజ్": ("damage", "LIGHT_VERB"),
    "మెసేజ్‌లు": ("messages-lu", "HYBRID_FUSED"),
    "రిప్లై": ("reply", "EN"),
    "ఇంగ్లీష్": ("english", "EN"),
    "ఘోస్ట్": ("ghost", "LIGHT_VERB"),
    "డెలులు": ("delulu", "EN"),
    "వైబ్": ("vibe", "LIGHT_VERB"),
    "హింట్": ("hint", "EN"),
    "వీకెండ్": ("weekend", "EN"),
    "ట్రిప్": ("trip", "EN"),
    "బిల్డప్": ("buildup", "EN"),
    "లాస్ట్": ("last", "EN"),
    "మినిట్": ("minute", "EN"),
    "హ్యాండ్": ("hand", "EN"),
    "మేనేజర్": ("manager", "EN"),
    "బాస్": ("boss", "EN"),
    "ఫ్రైడే": ("friday", "EN"),
    "ఈవినింగ్": ("evening", "EN"),
    "క్లయింట్": ("client", "EN"),
    "కాల్": ("call", "EN"),
    "కాల్స్": ("calls", "EN"),
    "మైండ్": ("mind", "EN"),
    "బ్లాక్": ("block", "LIGHT_VERB"),
    "బెటర్": ("better", "EN"),
    "ట్రాఫిక్": ("traffic", "EN"),
    "సిగ్నల్": ("signal", "EN"),
    "జామ్": ("jam", "EN"),
    "క్లాస్": ("class", "EN"),
    "గ్యారెంటీ": ("guarantee", "EN"),
    "ప్రాజెక్ట్": ("project", "EN"),
    "మీటింగ్": ("meeting", "EN"),
    "డెడ్‌లైన్": ("deadline", "EN"),
    "శాలరీ": ("salary", "EN"),
    "బ్రేకప్": ("breakup", "EN"),
    "ప్యాచప్": ("patchup", "EN"),
    "ఫోన్": ("phone", "EN"),
    "కార్": ("car", "EN"),
    "బైక్": ("bike", "EN"),
    "హోటల్": ("hotel", "EN"),
    "బిల్": ("bill", "EN"),
    "ఆర్డర్": ("order", "LIGHT_VERB"),
    "డాక్టర్": ("doctor", "EN"),
    "హాస్పిటల్": ("hospital", "EN"),
    "టికెట్": ("ticket", "EN"),
    "సినిమా": ("cinema", "EN"),
    "థియేటర్": ("theater", "EN"),
    "లవ్": ("love", "EN"),
    "హార్ట్": ("heart", "EN"),
    "బ్రెయిన్": ("brain", "EN"),
    "సార్": ("sir", "EN"),
    "మేడమ్": ("madam", "EN"),
    "షర్ట్": ("shirt", "EN"),
    "షూ": ("shoe", "EN"),
    "సైజు": ("size", "EN"),
    "కన్‌స్ట్రక్షన్": ("construction", "EN"),
    "కంపెనీ": ("company", "EN"),
    "బ్యాంక్": ("bank", "EN"),
    "లోన్": ("loan", "EN"),
    "శాంక్షన్": ("sanction", "LIGHT_VERB"),
    "టాలెంట్": ("talent", "EN"),
    "ఎంగేజ్‌మెంట్": ("engagement", "EN"),
    "ఫిక్స్": ("fix", "LIGHT_VERB"),
    "ఆల్రెడీ": ("already", "EN"),
    "ఐస్": ("ice", "EN"),
    "క్రీమ్": ("cream", "EN"),
    "కిక్": ("kick", "EN"),
    "సీక్రెట్": ("secret", "EN"),
    "టైమ్": ("time", "EN"),
    "టైం": ("time", "EN"),
    "హ్యాపీగా": ("happy-ga", "HYBRID_FUSED"),
    "హ్యాపీనెస్": ("happiness", "EN"),
    "లాంగ్": ("long", "EN"),
    "అంకుల్": ("uncle", "EN"),
    "ఆంటీ": ("aunty", "EN"),
    "ఫ్యామిలీ": ("family", "EN"),
    "సైలెంట్": ("silent", "EN"),
    "ఫైన్": ("fine", "EN"),
    "ప్లీజ్": ("please", "EN"),
    "సెట్": ("set", "LIGHT_VERB"),
    "రెస్ట్": ("rest", "LIGHT_VERB"),
    "కిచెన్": ("kitchen", "EN"),
    "సారీ": ("sorry", "EN"),
    "హెల్ప్": ("help", "LIGHT_VERB"),
    "మెస్": ("mess", "LIGHT_VERB"),
}

TELUGU_ROMAN_SUFFIXES = (
    "ki", "lo", "tho", "ni", "kosam", "nundi", "nudchi", "meeda", "la", "laga", "ga", "ra", "lu"
)

CANONICAL_TO_PERSONA_LEXICON: Dict[str, Dict[str, str]] = {
    "అంతేగా": {
        "P1_STANDARD": "anthega",
        "P2_MINIMAL": "antega",
        "P3_EXPRESS": "antheygaaa",
        "P4_ACROPHON": "anthe ga",
    },
    "ఏంటి": {
        "P1_STANDARD": "yenti",
        "P2_MINIMAL": "enti",
        "P3_EXPRESS": "yentiii",
        "P4_ACROPHON": "Anti",
    },
    "ఏంట్రా": {
        "P1_STANDARD": "yentra",
        "P2_MINIMAL": "entra",
        "P3_EXPRESS": "yentiraaa",
        "P4_ACROPHON": "Antra",
    },
    "ఏంది": {
        "P1_STANDARD": "yendhi",
        "P2_MINIMAL": "endi",
        "P3_EXPRESS": "yendhii",
        "P4_ACROPHON": "Andi",
    },
    "ఏందిరా": {
        "P1_STANDARD": "yendhira",
        "P2_MINIMAL": "endira",
        "P3_EXPRESS": "yendhiraaa",
        "P4_ACROPHON": "Andira",
    },
    "ఏం": {
        "P1_STANDARD": "yem",
        "P2_MINIMAL": "em",
        "P3_EXPRESS": "yemmm",
        "P4_ACROPHON": "m",
    },
    "ఎందుకు": {
        "P1_STANDARD": "endhuku",
        "P2_MINIMAL": "enduku",
        "P3_EXPRESS": "yendhukuuu",
        "P4_ACROPHON": "nduku",
    },
    "ఎలా": {
        "P1_STANDARD": "yela",
        "P2_MINIMAL": "ela",
        "P3_EXPRESS": "yelaaa",
        "P4_ACROPHON": "la",
    },
    "ఎక్కడ": {
        "P1_STANDARD": "ekkada",
        "P2_MINIMAL": "ekada",
        "P3_EXPRESS": "yekkada",
        "P4_ACROPHON": "kkada",
    },
    "ఎక్కడికి": {
        "P1_STANDARD": "ekkadiki",
        "P2_MINIMAL": "ekadiki",
        "P3_EXPRESS": "yekkadikiii",
        "P4_ACROPHON": "kkadiki",
    },
    "చెప్పు": {
        "P1_STANDARD": "cheppu",
        "P2_MINIMAL": "chepu",
        "P3_EXPRESS": "cheppuuu",
        "P4_ACROPHON": "chappu",
    },
    "చెప్తున్నావ్": {
        "P1_STANDARD": "chepthunnav",
        "P2_MINIMAL": "cheptunav",
        "P3_EXPRESS": "chepthunnava",
        "P4_ACROPHON": "chptnv",
    },
    "చేస్తున్నావ్": {
        "P1_STANDARD": "chesthunnav",
        "P2_MINIMAL": "chestunav",
        "P3_EXPRESS": "chesthunnaav",
        "P4_ACROPHON": "chstnv",
    },
    "చేస్తున్నావు": {
        "P1_STANDARD": "chesthunnavu",
        "P2_MINIMAL": "chestunav",
        "P3_EXPRESS": "chesthunnaav",
        "P4_ACROPHON": "chstnv",
    },
    "నువ్వు": {
        "P1_STANDARD": "nuvvu",
        "P2_MINIMAL": "nuvu",
        "P3_EXPRESS": "nuvvuuu",
        "P4_ACROPHON": "nv",
    },
    "నాకు": {
        "P1_STANDARD": "naaku",
        "P2_MINIMAL": "naku",
        "P3_EXPRESS": "naakuuu",
        "P4_ACROPHON": "nku",
    },
    "నీకు": {
        "P1_STANDARD": "neeku",
        "P2_MINIMAL": "niku",
        "P3_EXPRESS": "neekuuu",
        "P4_ACROPHON": "nku",
    },
    "అర్థం": {
        "P1_STANDARD": "ardham",
        "P2_MINIMAL": "artam",
        "P3_EXPRESS": "artham",
        "P4_ACROPHON": "ardhm",
    },
    "కావట్లేదు": {
        "P1_STANDARD": "kaavatledhu",
        "P2_MINIMAL": "kavatledu",
        "P3_EXPRESS": "kaavatleee",
        "P4_ACROPHON": "kavatle",
    },
    "అవ్వట్లేదు": {
        "P1_STANDARD": "avvatledhu",
        "P2_MINIMAL": "avvatledu",
        "P3_EXPRESS": "avvatleee",
        "P4_ACROPHON": "avtle",
    },
    "ఇవ్వట్లేదు": {
        "P1_STANDARD": "ivvatledhu",
        "P2_MINIMAL": "ivvatledu",
        "P3_EXPRESS": "ivvatleee",
        "P4_ACROPHON": "ivtle",
    },
    "అసలు": {
        "P1_STANDARD": "asalu",
        "P2_MINIMAL": "aslu",
        "P3_EXPRESS": "assalu",
        "P4_ACROPHON": "aslu",
    },
    "ఇప్పుడు": {
        "P1_STANDARD": "ippudu",
        "P2_MINIMAL": "ipudu",
        "P3_EXPRESS": "ippuduuu",
        "P4_ACROPHON": "ipdu",
    },
    "తెలుసు": {
        "P1_STANDARD": "thelusu",
        "P2_MINIMAL": "telusu",
        "P3_EXPRESS": "thelusU",
        "P4_ACROPHON": "talusu",
    },
    "తెలుసా": {
        "P1_STANDARD": "thelusaa",
        "P2_MINIMAL": "telsa",
        "P3_EXPRESS": "thelusaaa",
        "P4_ACROPHON": "talsa",
    },
    "యుద్ధంలో": {
        "P1_STANDARD": "yuddhamlo",
        "P2_MINIMAL": "yudhamlo",
        "P3_EXPRESS": "yuddham lo",
        "P4_ACROPHON": "yudhamlo",
    },
}

REVERSE_VARIANT_TO_CANONICAL: Dict[str, str] = {}
for canonical_te, p_map in CANONICAL_TO_PERSONA_LEXICON.items():
    for _, variant in p_map.items():
        REVERSE_VARIANT_TO_CANONICAL[variant.lower()] = canonical_te

EXTRA_VARIANTS = {
    "anteyga": "అంతేగా",
    "anthayga": "అంతేగా",
    "anthegaa": "అంతేగా",
    "anti": "ఏంటి",
    "antra": "ఏంట్రా",
    "ndku": "ఎందుకు",
    "talusu": "తెలుసు",
    "telsu": "తెలుసు",
    "thelusu": "తెలుసు",
    "vallu": "వెళ్ళు",
    "yudhamlo": "యుద్ధంలో",
    "yuddhamlo": "యుద్ధంలో",
}
REVERSE_VARIANT_TO_CANONICAL.update(EXTRA_VARIANTS)


def disambiguate_homograph(word: str, prev_word: str = "", next_word: str = "") -> Dict[str, Any]:
    """Resolves cross-language homographs such as 'Anti'/'anti', 'm', 'la', 'pattu'."""
    w_low = word.strip("?!.,:;").lower()
    p_low = prev_word.strip("?!.,:;").lower()
    n_low = next_word.strip("?!.,:;").lower()

    if w_low == "anti":
        if p_low in {"maa", "ma", "mee", "mi", "pakkinti", "colony", "sari", "sorry"} or n_low in {"vachindi", "cheppindi", "intlo", "iroju", "eeroju"}:
            return {
                "surface": word,
                "resolved_te": "ఆంటీ",
                "meaning": "Aunty (Loanword Noun)",
                "confidence": 0.94,
                "escalated_to_archon": True,
            }
        if n_low in {"social", "hero", "national", "virus", "corruption", "biotic"}:
            return {
                "surface": word,
                "resolved_te": "యాంటీ (anti-)",
                "meaning": "English prefix 'anti-'",
                "confidence": 0.96,
                "escalated_to_archon": True,
            }
        return {
            "surface": word,
            "resolved_te": "ఏంటి",
            "meaning": "What (Gen-Z Acrophonic 'A' + 'nti')",
            "confidence": 0.95,
            "escalated_to_archon": True,
        }

    if w_low == "m":
        if n_low in {"chestunnav", "chstnv", "aindhi", "aindi", "cheppu", "chappu", "ledhu", "ledu", "plan", "adigina", "nachchindi"} or p_low in {"asalu", "aslu", "bro", "ippudu", "ipdu", "nalo"}:
            return {
                "surface": word,
                "resolved_te": "ఏం",
                "meaning": "What ('M' letter-name acrophony for ఏం)",
                "confidence": 0.93,
                "escalated_to_archon": True,
            }

    if w_low in REVERSE_VARIANT_TO_CANONICAL:
        return {
            "surface": word,
            "resolved_te": REVERSE_VARIANT_TO_CANONICAL[w_low],
            "meaning": "Canonical Telugu match",
            "confidence": 0.98,
            "escalated_to_archon": False,
        }

    return {
        "surface": word,
        "resolved_te": word,
        "meaning": "Pass-through token",
        "confidence": 0.85,
        "escalated_to_archon": False,
    }


def classify_token(token: str) -> Tuple[str, str]:
    """Classifies token into EN, HYBRID_FUSED, LIGHT_VERB, or TE."""
    clean_tok = re.sub(r"[^\w\u0c00-\u0c7f-]", "", token)
    if not clean_tok:
        return ("PUNCT", "")

    if clean_tok in TELUGU_SCRIPT_ENGLISH_LOANS:
        en_equiv, tag_type = TELUGU_SCRIPT_ENGLISH_LOANS[clean_tok]
        return (tag_type, f"{clean_tok} -> {en_equiv}")

    for loan_te, (en_equiv, _) in TELUGU_SCRIPT_ENGLISH_LOANS.items():
        if clean_tok.startswith(loan_te) and len(clean_tok) > len(loan_te):
            suffix_te = clean_tok[len(loan_te) :]
            suffix_rom = transliterate_telugu_word_universal(suffix_te, "P2_MINIMAL")
            return ("HYBRID_FUSED", f"{en_equiv}+-{suffix_rom}")

    low_tok = clean_tok.lower()

    if "-" in low_tok:
        parts = low_tok.split("-", 1)
        if parts[0] in ENGLISH_CORE_WORDS and parts[1] in TELUGU_ROMAN_SUFFIXES:
            return ("HYBRID_FUSED", f"{parts[0]}+-{parts[1]}")

    if low_tok not in ENGLISH_CORE_WORDS and low_tok not in REVERSE_VARIANT_TO_CANONICAL:
        for suf in TELUGU_ROMAN_SUFFIXES:
            if low_tok.endswith(suf) and len(low_tok) > len(suf) + 2:
                stem = low_tok[: -len(suf)]
                if stem in ENGLISH_CORE_WORDS:
                    return ("HYBRID_FUSED", f"{stem}+-{suf}")

    if low_tok in ENGLISH_CORE_WORDS:
        return ("EN", low_tok)

    return ("TE", clean_tok)


def compute_utterance_cmi(text: str) -> Dict[str, Any]:
    """Computes Code-Mixing Index (CMI %) and counts of EN, TE, HYBRID_FUSED, LIGHT_VERB tokens."""
    raw_tokens = [t for t in text.split() if re.sub(r"[^\w\u0c00-\u0c7f]", "", t)]
    counts = Counter()
    tagged_tokens = []

    for tok in raw_tokens:
        cat, detail = classify_token(tok)
        if cat == "PUNCT":
            continue
        counts[cat] += 1
        tagged_tokens.append({"token": tok, "tag": cat, "detail": detail})

    total = sum(counts.values())
    if total == 0:
        return {"cmi_pct": 0.0, "en_lexeme_pct": 0.0, "counts": dict(counts), "tagged_tokens": []}

    en_total = counts["EN"] + counts["HYBRID_FUSED"] + counts["LIGHT_VERB"]
    te_total = counts["TE"]
    en_ratio_pct = round(100.0 * en_total / total, 2)
    cmi_formula = round(100.0 * (1.0 - max(en_total, te_total) / total), 2)

    return {
        "total_words": total,
        "en_lexeme_pct": en_ratio_pct,
        "cmi_gamback": cmi_formula,
        "counts": {
            "TE": counts["TE"],
            "EN": counts["EN"],
            "HYBRID_FUSED": counts["HYBRID_FUSED"],
            "LIGHT_VERB": counts["LIGHT_VERB"],
        },
        "tagged_tokens": tagged_tokens,
    }


# Only non-emotional functional/conversational nouns are eligible for synthetic English swap;
# PROTECTED_TELUGU_EMOTIONAL_CORE words ('kopam', 'nijam', 'dhairyam', 'prema', 'baadha') NEVER swap!
ERA_LEXICAL_SWAPS = {
    "విషయం": {"ERA_2000": "matter", "ERA_2025": "matter"},
    "ప్రయత్నించాను": {"ERA_2000": "try chesanu", "ERA_2025": "try chesna"},
    "పరిస్థితులు": {"ERA_2000": "situations", "ERA_2025": "situations"},
    "సమస్య": {"ERA_2000": "problem", "ERA_2025": "scene"},
    "ఆలోచిస్తే": {"ERA_2000": "think cheste", "ERA_2025": "overthink cheste"},
}


def romanize_with_persona(te_sentence: str, persona: str = "P1_STANDARD", era_mode: str = "ERA_2025") -> str:
    """
    Converts ANY Telugu sentence into 100% Latin-script Tenglish according to the chosen
    Texter Persona while:
    - Keeping PROTECTED_TELUGU_EMOTIONAL_CORE words ('kopam', 'nijam', 'dhairyam', 'yuddhamlo') in Telugu
    - Keeping NATURAL_MOVIE_ENGLISH_LEXICON words ('life', 'career', 'coffee', 'settle') in English.
    """
    tokens = te_sentence.split()
    out_tokens = []

    for tok in tokens:
        punct = ""
        m = re.search(r"([?!.,:;\"']+$)", tok)
        clean_w = tok
        if m:
            punct = m.group(1)
            clean_w = tok[: -len(punct)]

        # 1. High-frequency Multi-Persona Lexicon ('anthega'/'antega', 'Anti'=ఏంటి, 'yuddhamlo')
        if clean_w in CANONICAL_TO_PERSONA_LEXICON:
            mapped = CANONICAL_TO_PERSONA_LEXICON[clean_w].get(persona, clean_w)
            out_tokens.append(mapped + punct)
            continue

        # 2. Protected Telugu Emotional Core -> ALWAYS keep in Telugu (transliterated to Latin Tenglish)
        if clean_w in PROTECTED_TELUGU_EMOTIONAL_CORE:
            rom_emotional = transliterate_telugu_word_universal(clean_w, persona)
            out_tokens.append(rom_emotional + punct)
            continue

        # 3. Functional Era-based English Lexical Swap (non-emotional words like 'విషయం' -> 'matter')
        if clean_w in ERA_LEXICAL_SWAPS and era_mode in ERA_LEXICAL_SWAPS[clean_w]:
            out_tokens.append(ERA_LEXICAL_SWAPS[clean_w][era_mode] + punct)
            continue

        # 4. Words naturally used in English in Telugu movies ('life', 'career', 'coffee', 'office-ki')
        if clean_w in TELUGU_SCRIPT_ENGLISH_LOANS:
            out_tokens.append(TELUGU_SCRIPT_ENGLISH_LOANS[clean_w][0] + punct)
            continue

        matched_loan_prefix = False
        for loan_te, (en_equiv, _) in TELUGU_SCRIPT_ENGLISH_LOANS.items():
            if clean_w.startswith(loan_te) and len(clean_w) > len(loan_te):
                suffix_te = clean_w[len(loan_te) :]
                suffix_rom = transliterate_telugu_word_universal(suffix_te, persona)
                out_tokens.append(f"{en_equiv}-{suffix_rom}" + punct)
                matched_loan_prefix = True
                break
        if matched_loan_prefix:
            continue

        # 5. Universal Akshara-to-Latin Phonetic State Machine
        rom = transliterate_telugu_word_universal(clean_w, persona)
        out_tokens.append(rom + punct)

    return " ".join(out_tokens)


def expand_scene_across_eras_and_personas(scene_dict: Dict[str, Any]) -> Dict[str, Any]:
    """Expands 1 Core Telugu Scene across 4 Tenglish Personas with 100% Latin script coverage."""
    turns = scene_dict["turns"]
    expansions = {
        "ORIGINAL_TELUGU": turns,
        "P1_STANDARD_TENGLISH": [romanize_with_persona(t, "P1_STANDARD", "ERA_2025") for t in turns],
        "P2_MINIMAL_TENGLISH": [romanize_with_persona(t, "P2_MINIMAL", "ERA_2025") for t in turns],
        "P3_EXPRESSIVE_TENGLISH": [romanize_with_persona(t, "P3_EXPRESS", "ERA_2025") for t in turns],
        "P4_ACROPHONIC_BROKEN": [romanize_with_persona(t, "P4_ACROPHON", "ERA_2025") for t in turns],
    }
    return {
        "scene_id": scene_dict["scene_id"],
        "era": scene_dict["era"],
        "director_writer_style": scene_dict["director_writer_style"],
        "emotion_hint": scene_dict["emotion_hint"],
        "dialect_hint": scene_dict["dialect_hint"],
        "expansions": expansions,
    }


def calculate_spelling_entropy(variant_counts: Dict[str, int]) -> float:
    """Computes Shannon Entropy H(X) = -sum(p_i * log2(p_i)) in bits."""
    total = sum(variant_counts.values())
    if total == 0:
        return 0.0
    entropy = 0.0
    for c in variant_counts.values():
        if c > 0:
            p = c / total
            entropy -= p * math.log2(p)
    return round(entropy, 3)


# --- Movie Harvester & SRT Parser ---
"""
Milestone 1 (Day 1): Core Telugu Movie & Literature Seed Harvester.

Provides:
1. SRT / YouTube caption parser with pause-gap turn diarization (delta_t > 1.5s = new speaker turn)
2. Automated ASR / BGM noise cleaner & copyright-safe 2-to-4 turn scene slicer
3. yt-dlp wrapper for Colab CPU harvesting from YouTube Telugu movie channels
4. Rich curated Multi-Era Seed Corpus (1980s -> 2020s + Literature) spanning Director/Writer styles,
   Genres, Emotions, and Regional Dialects for immediate baseline calibration.
"""

import json
import re
import subprocess
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional


@dataclass
class SubtitleBlock:
    index: int
    start_sec: float
    end_sec: float
    text: str


@dataclass
class DialogueScene:
    scene_id: str
    era: str
    source_type: str  # "youtube_movie" | "literature" | "short_film"
    genre_hint: str
    director_writer_style: str
    dialect_hint: str
    emotion_hint: str
    turns: List[str]
    metadata: Dict[str, Any] = field(default_factory=dict)


NOISE_PATTERNS = [
    re.compile(r"\[(?:music|applause|laughter|bgm|silence|noise|singing).*?\]", re.IGNORECASE),
    re.compile(r"\((?:music|bgm|applause|sighs|laughs).*?\)", re.IGNORECASE),
    re.compile(r"[♪♫♬]+"),
    re.compile(r"<[^>]+>"),
    re.compile(r"\b(?:la\s+){3,}\b", re.IGNORECASE),
]

USER_HANDLE_PATTERN = re.compile(r"(?:@[\w_]+|u/[\w_-]+)")


def clean_utterance(text: str) -> str:
    """Strips BGM tags, HTML tags, music symbols, and user handles."""
    cleaned = text.strip()
    for pat in NOISE_PATTERNS:
        cleaned = pat.sub("", cleaned)
    cleaned = USER_HANDLE_PATTERN.sub("[USER]", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def parse_timestamp_to_seconds(ts: str) -> float:
    """Parses SRT timestamp 'HH:MM:SS,mmm' or 'HH:MM:SS.mmm' to seconds."""
    ts = ts.strip().replace(",", ".")
    parts = ts.split(":")
    if len(parts) != 3:
        return 0.0
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])


def parse_srt_content(srt_text: str) -> List[SubtitleBlock]:
    """Parses raw .srt file content into structured SubtitleBlock items."""
    blocks: List[SubtitleBlock] = []
    raw_chunks = re.split(r"\n\s*\n", srt_text.strip())
    for idx, chunk in enumerate(raw_chunks, start=1):
        lines = [line.strip() for line in chunk.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        ts_line_idx = 0
        for i, l in enumerate(lines):
            if "-->" in l:
                ts_line_idx = i
                break
        else:
            continue

        ts_parts = lines[ts_line_idx].split("-->")
        start_s = parse_timestamp_to_seconds(ts_parts[0])
        end_s = parse_timestamp_to_seconds(ts_parts[1])
        text_body = clean_utterance(" ".join(lines[ts_line_idx + 1 :]))
        if text_body and len(text_body) >= 2:
            blocks.append(SubtitleBlock(index=idx, start_sec=start_s, end_sec=end_s, text=text_body))
    return blocks


def slice_blocks_into_scenes(
    blocks: List[SubtitleBlock],
    era: str,
    genre_hint: str,
    director_writer_style: str,
    dialect_hint: str,
    emotion_hint: str,
    scene_prefix: str = "MOV",
    same_turn_gap_sec: float = 1.2,
    new_scene_gap_sec: float = 4.5,
    max_turns_per_scene: int = 4,
) -> List[DialogueScene]:
    """
    Groups subtitle blocks into 2-to-4 turn conversational exchanges based on timing gaps.
    Enforces copyright-safe max_turns_per_scene <= 4.
    """
    if not blocks:
        return []

    scenes: List[DialogueScene] = []
    current_turns: List[str] = []
    current_turn_buffer: List[str] = [blocks[0].text]
    prev_end = blocks[0].end_sec
    scene_counter = 1

    for b in blocks[1:]:
        gap = b.start_sec - prev_end
        if gap <= same_turn_gap_sec:
            current_turn_buffer.append(b.text)
        elif gap <= new_scene_gap_sec and len(current_turns) < max_turns_per_scene - 1:
            current_turns.append(" ".join(current_turn_buffer))
            current_turn_buffer = [b.text]
        else:
            current_turns.append(" ".join(current_turn_buffer))
            if len(current_turns) >= 2:
                scenes.append(
                    DialogueScene(
                        scene_id=f"{scene_prefix}_{era}_{scene_counter:04d}",
                        era=era,
                        source_type="youtube_movie",
                        genre_hint=genre_hint,
                        director_writer_style=director_writer_style,
                        dialect_hint=dialect_hint,
                        emotion_hint=emotion_hint,
                        turns=current_turns[:max_turns_per_scene],
                    )
                )
                scene_counter += 1
            current_turns = []
            current_turn_buffer = [b.text]
        prev_end = b.end_sec

    if current_turn_buffer:
        current_turns.append(" ".join(current_turn_buffer))
    if len(current_turns) >= 2:
        scenes.append(
            DialogueScene(
                scene_id=f"{scene_prefix}_{era}_{scene_counter:04d}",
                era=era,
                source_type="youtube_movie",
                genre_hint=genre_hint,
                director_writer_style=director_writer_style,
                dialect_hint=dialect_hint,
                emotion_hint=emotion_hint,
                turns=current_turns[:max_turns_per_scene],
            )
        )
    return scenes


def harvest_youtube_subtitles(
    youtube_url: str,
    output_dir: str | Path,
    langs: str = "te,en",
) -> List[Path]:
    """
    Downloads official/manual subtitles from a YouTube video using yt-dlp (Colab CPU friendly).
    """
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "yt-dlp",
        "--write-sub",
        "--write-auto-sub",
        "--sub-lang",
        langs,
        "--sub-format",
        "srt",
        "--skip-download",
        "-o",
        str(out_dir / "%(id)s.%(ext)s"),
        youtube_url,
    ]
    subprocess.run(cmd, check=False, capture_output=True, text=True)
    return sorted(out_dir.glob("*.srt"))


def build_curated_multi_era_seed_corpus() -> List[DialogueScene]:
    """
    Returns an authentic, multi-era Telugu Cinema & Literature Seed Corpus (1980s -> 2020s)
    covering 6 iconic Writer/Director archetypes, 6 emotions, and 5 dialects.
    Used to calibrate historical English % (CMI) baselines and feed the Tenglish-Evo Multiplier.
    """
    raw_seed = [
        # --- ERA 0: CLASSIC LITERATURE & FOLK PROSE (0% English Baseline) ---
        {
            "scene_id": "SEED_LIT_0001",
            "era": "1980s_LITERARY",
            "source_type": "literature",
            "genre_hint": "FAMILY_DRAMA",
            "director_writer_style": "CLASSIC_LITERARY",
            "dialect_hint": "STANDARD_TELUGU",
            "emotion_hint": "PHILOSOPHICAL",
            "turns": [
                "మనిషికి మాటే ప్రాణం అబ్బాయ్, ఆ మాట తప్పితే ఇక విలువ ఏముంటుంది?",
                "కానీ పరిస్థితులు మారినప్పుడు మనుషులు కూడా మారుతున్నారు కదా తాతయ్య.",
                "పరిస్థితులకు తలొగ్గేవాడు సామాన్యుడు, కష్టాల్లో కూడా నిజాయితీగా నిలబడేవాడే గొప్పవాడు.",
            ],
        },
        {
            "scene_id": "SEED_LIT_0002",
            "era": "1980s_LITERARY",
            "source_type": "literature",
            "genre_hint": "SATIRE_COMEDY",
            "director_writer_style": "JANDHYALA_WORDPLAY",
            "dialect_hint": "COASTAL_ANDHRA",
            "emotion_hint": "COMIC_TIMING",
            "turns": [
                "ఏమండోయ్ శాస్త్రి గారూ, తమరు పొద్దున్నే ఇలా హడావిడిగా ఎక్కడికి బయలుదేరారు?",
                "ఏం చెప్పమంటారు సుబ్బారావు గారూ, మా ఆవిడ పుట్టింటికి వెళ్తానని భీష్మించుకొని కూర్చుంది!",
                "అయ్యో, అది సంతోషించాల్సిన విషయమే కదండీ, ఇంతకీ రైలు ఖర్చులకు డబ్బులు ఇచ్చారా లేదా?",
            ],
        },
        # --- ERA 1: 1980s-1990s CLASSIC CINEMA (Low English Loanwords ~2-5%) ---
        {
            "scene_id": "SEED_1990_0001",
            "era": "1990s",
            "source_type": "youtube_movie",
            "genre_hint": "SATIRE_COMEDY",
            "director_writer_style": "JANDHYALA_WORDPLAY",
            "dialect_hint": "COASTAL_ANDHRA",
            "emotion_hint": "COMIC_TIMING",
            "turns": [
                "ఏరా అబ్బాయ్, కాలేజీకి వెళ్ళకుండా ఇక్కడ బస్ స్టాప్ దగ్గర ఏం చేస్తున్నావు?",
                "అది కాదండీ నాన్నగారూ, మా లెక్చరర్ గారు ఈరోజు సెలవు పెట్టారు.",
                "ఆహా! ఆయన సెలవు పెడితే నువ్వు పుస్తకాలు మూసేసి ఇలా రోడ్ల మీద తిరుగుతావా?",
            ],
        },
        {
            "scene_id": "SEED_1990_0002",
            "era": "1990s",
            "source_type": "youtube_movie",
            "genre_hint": "MASS_ACTION",
            "director_writer_style": "RAYALASEEMA_MASS",
            "dialect_hint": "RAYALASEEMA",
            "emotion_hint": "ANGRY_WARNING",
            "turns": [
                "ఏందిరా, మా ఊరి పొలిమేరల్లోకి వచ్చి మాకే ఎదురు తిరుగుతావా?",
                "న్యాయం ఎక్కడుంటే అక్కడికి రావడానికి నాకు ఎవ్వడి పర్మిషన్ అక్కర్లేదు.",
                "ఒక్కసారి కత్తి పడితే వెనక్కి తగ్గే రక్తం కాదురా ఇది, జాగ్రత్త!",
            ],
        },
        {
            "scene_id": "SEED_1990_0003",
            "era": "1990s",
            "source_type": "youtube_movie",
            "genre_hint": "FAMILY_DRAMA",
            "director_writer_style": "CLASSIC_DRAMA",
            "dialect_hint": "STANDARD_TELUGU",
            "emotion_hint": "HEARTBREAK",
            "turns": [
                "ఇన్నాళ్లూ మన మధ్య ఉన్న స్నేహం అంతా అబద్ధమేనా? నాకు ఒక్క మాట కూడా చెప్పాలనిపించలేదా?",
                "చెప్పాలని చాలాసార్లు ప్రయత్నించాను రా, కానీ నీ ముఖం చూసి నిజం చెప్పే ధైర్యం రాలేదు.",
                "నిజం దాచిపెట్టి నన్ను మోసం చేయడం కంటే, ఆ రోజే నన్ను వదిలేసి వెళ్ళిపోవాల్సింది.",
            ],
        },
        # --- ERA 2: 2000s URBAN & TRIVIKRAM / PURI ERA (~12-22% English Mix) ---
        {
            "scene_id": "SEED_2000_0001",
            "era": "2000s",
            "source_type": "youtube_movie",
            "genre_hint": "ROMCOM",
            "director_writer_style": "TRIVIKRAM_ANTITHESIS",
            "dialect_hint": "STANDARD_TELUGU",
            "emotion_hint": "PHILOSOPHICAL",
            "turns": [
                "నిజం చెప్పాలంటే నాకు తన మీద కోపం లేదు రా, కానీ తనని మర్చిపోవడం మాత్రం నా వల్ల కావట్లేదు.",
                "ప్రేమించేటప్పుడు ఉన్న ధైర్యం, ఆ ప్రేమని నిలబెట్టుకునేటప్పుడు ఎందుకు ఉండదు రా మీకు?",
                "యుద్ధంలో గెలవడం అంటే శత్రువుని చంపడం కాదు... మనల్ని మనం గెలవడం. ప్రేమలో ఓడిపోయినా పర్లేదు రా.",
            ],
        },
        {
            "scene_id": "SEED_2000_0002",
            "era": "2000s",
            "source_type": "youtube_movie",
            "genre_hint": "MASS_ACTION",
            "director_writer_style": "PURI_CYNICAL_PUNCH",
            "dialect_hint": "URBAN_HYDERABAD",
            "emotion_hint": "ELEVATION_PUNCH",
            "turns": [
                "ఏంటి భయ్యా, పోలీస్ స్టేషన్ లో కూర్చొని కూడా ఇంత కూల్ గా కాఫీ తాగుతున్నావ్?",
                "నాకు భయం అనే ఫీలింగ్ చిన్నప్పుడే ఆపరేషన్ చేసి తీసేశారు రా.",
                "నేను ఒక్కసారి కమిట్ అయితే నా మాట నేనే వినను, ఇక నీ వార్నింగ్ ఎవడు వింటాడు?",
            ],
        },
        {
            "scene_id": "SEED_2000_0003",
            "era": "2000s",
            "source_type": "youtube_movie",
            "genre_hint": "FAMILY_DRAMA",
            "director_writer_style": "BOMMARILLU_EMOTIONAL",
            "dialect_hint": "URBAN_HYDERABAD",
            "emotion_hint": "HEARTBREAK",
            "turns": [
                "నాన్న, నా లైఫ్ లో ఏ డెసిషన్ అయినా నేనే తీసుకోవాలని ఎప్పుడైనా అడిగానా?",
                "నీ మంచి కోసమే కదరా ఇదంతా చేస్తున్నాను, నీ కెరీర్ సెటిల్ అవ్వాలని నా తాపత్రయం.",
                "మొత్తం మీరే డిసైడ్ చేసేస్తే ఇక నేను బతకడం ఎందుకు నాన్న? నా ఫీలింగ్స్ కి అసలు వాల్యూ లేదా?",
            ],
        },
        # --- ERA 3: 2010s MULTIPLEX & LOGIC/NEW-WAVE ERA (~22-32% English Mix) ---
        {
            "scene_id": "SEED_2010_0001",
            "era": "2010s",
            "source_type": "youtube_movie",
            "genre_hint": "THRILLER",
            "director_writer_style": "SUKUMAR_LOGIC",
            "dialect_hint": "URBAN_HYDERABAD",
            "emotion_hint": "ELEVATION_PUNCH",
            "turns": [
                "వాడు వేసిన ప్రతి ప్లాన్ వెనుక ఒక క్యాలిక్యులేషన్ ఉంది సార్, మనం నార్మల్ గా ఆలోచిస్తే దొరకడు.",
                "అంటే ఇప్పుడు వాడు మన నెక్స్ట్ మూవ్ ని కూడా ముందే గెస్ చేశాడంటావా?",
                "ఎగ్జాక్ట్‌లీ! వాడు ఎమోషన్ తో కాదు, ప్యూర్ లాజిక్ తో గేమ్ ఆడుతున్నాడు.",
            ],
        },
        {
            "scene_id": "SEED_2010_0002",
            "era": "2010s",
            "source_type": "youtube_movie",
            "genre_hint": "ROMCOM",
            "director_writer_style": "THARUN_TELANGANA_BANTER",
            "dialect_hint": "TELANGANA",
            "emotion_hint": "SARCASTIC_ROAST",
            "turns": [
                "ఏంది రా బై, జాబ్ వదిలేసి బిజినెస్ స్టార్ట్ చేస్తా అంటున్నవ్, అసలు ప్లాన్ ఏమన్నా ఉందా?",
                "ప్లాన్ గీన్ ఏం లేదు రా, ఫస్ట్ దిగినంకనే స్విమ్మింగ్ వస్తది అని మా దోస్తుగాడు చెప్పిండు.",
                "నీ దోస్తుగాని మాటలు విని దిగితే స్విమ్మింగ్ కాదు రా, డైరెక్ట్ మునిగిపోతవ్! లైట్ తీసుకో.",
            ],
        },
        {
            "scene_id": "SEED_2010_0003",
            "era": "2010s",
            "source_type": "youtube_movie",
            "genre_hint": "ROMCOM",
            "director_writer_style": "TRIVIKRAM_ANTITHESIS",
            "dialect_hint": "STANDARD_TELUGU",
            "emotion_hint": "SARCASTIC_ROAST",
            "turns": [
                "ఏంట్రా, ఆ అమ్మాయి నిన్ను రిజెక్ట్ చేసిందని పొద్దున్నుంచి ఆఫీస్ కి రాకుండా రూమ్ లోనే కూర్చున్నావా?",
                "తనతో కనెక్ట్ అయ్యాను రా, ఇప్పుడు సడన్ గా కుదరదు అంటే ఎలా తీసుకోవాలో అర్థం కావట్లేదు.",
                "అవసరం ఉన్నప్పుడు గుర్తొచ్చేది ఆప్షన్ రా, ఊపిరి ఉన్నంతవరకు గుర్తుండేది ఎమోషన్. తను నిన్ను ఆప్షన్ లా చూసింది, వదిలేయ్!",
            ],
        },
        # --- ERA 4: 2020s GEN-Z / DJ TILLU / VIVEK ATHREYA ERA (~35-50% English Mix) ---
        {
            "scene_id": "SEED_2020_0001",
            "era": "2020s",
            "source_type": "youtube_movie",
            "genre_hint": "SATIRE_COMEDY",
            "director_writer_style": "THARUN_TELANGANA_BANTER",
            "dialect_hint": "TELANGANA",
            "emotion_hint": "COMIC_TIMING",
            "turns": [
                "ఏంది బ్రో ఇది, నిన్న రాత్రి పార్టీకి పిలిచి ఇప్పుడు నాకు సంబంధం లేదు అంటావ్ ఏంది?",
                "అరే భయ్యా, అక్కడ సిట్యుయేషన్ మొత్తం కంట్రోల్ తప్పింది రా, నాకే ఫుల్ ట్రామా అయిపోయింది!",
                "నువ్వు ఇట్లా ప్రతిసారి సీన్ క్రియేట్ చేసి నా మీద తోసేస్తే నా లైఫ్ మొత్తం డ్యామేజ్ అయితది అంతేగా!",
            ],
        },
        {
            "scene_id": "SEED_2020_0002",
            "era": "2020s",
            "source_type": "youtube_movie",
            "genre_hint": "ROMCOM",
            "director_writer_style": "VIVEK_URBAN_WIT",
            "dialect_hint": "URBAN_HYDERABAD",
            "emotion_hint": "SARCASTIC_ROAST",
            "turns": [
                "బ్రో, నిన్నటి నుంచి తనకి పది మెసేజ్‌లు పెట్టాను, సీన్ చేసింది కానీ అసలు రిప్లై ఇవ్వట్లేదు ఏంటి?",
                "దాన్నే ఇంగ్లీష్ లో ఘోస్ట్ చేయడం అంటారు రా, ఇంకా ఆ డెలులు లో నుంచి బయటికి రా!",
                "అంటే ఇప్పుడు మనకి అక్కడ అసలు వైబ్ అవ్వట్లేదు అని డైరెక్ట్ గా హింట్ ఇచ్చేసిందా?",
            ],
        },
        {
            "scene_id": "SEED_2020_0003",
            "era": "2020s",
            "source_type": "youtube_movie",
            "genre_hint": "SATIRE_COMEDY",
            "director_writer_style": "GENZ_MEME_BANTER",
            "dialect_hint": "URBAN_HYDERABAD",
            "emotion_hint": "SARCASTIC_ROAST",
            "turns": [
                "ఏంటి బ్రో, వీకెండ్ ట్రిప్ కి వస్తా అని ఫుల్ బిల్డప్ ఇచ్చి లాస్ట్ మినిట్ లో హ్యాండ్ ఇచ్చావ్?",
                "మా మేనేజర్ గాడు ఫ్రైడే ఈవినింగ్ క్లయింట్ కాల్ పెట్టి మైండ్ బ్లాక్ చేసిండు రా, నేనేం చేయను చెప్పు?",
                "ప్రతిసారి ఇదే సాకు చెప్తున్నావ్ రా, ఇంక నీతో ట్రిప్ ప్లాన్ చేయడం కంటే ఇంట్లో పడుకోవడం బెటర్ అంతేగా!",
            ],
        },
    ]

    return [DialogueScene(**item) for item in raw_seed]


def save_scenes_jsonl(scenes: List[DialogueScene], out_path: str | Path) -> Path:
    """Writes DialogueScene list to JSONL format."""
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        for s in scenes:
            f.write(json.dumps(asdict(s), ensure_ascii=False) + "\n")
    return p


# --- YouTube Extractor & CSV/TXT Exporter ---
"""
YouTube Full-Movie Extractor & Auto-Tagger CLI (`extract_youtube_movie.py`).

Usage (in Google Colab or local terminal):
  python3 extract_youtube_movie.py \
      --url "https://www.youtube.com/watch?v=..." \
      --movie-name "Bommarillu_2006" \
      --era "2000s" \
      --director-style "BHASKAR_ABBURI_RAVI" \
      --out-dir "./drive_mirror/Tenglish_LM/extracted_movies"

Or test on a local .srt file:
  python3 extract_youtube_movie.py \
      --srt-file "./drive_mirror/Tenglish_LM/bommarillu_2006/bommarillu_2006.srt" \
      --movie-name "Bommarillu_2006" \
      --era "2000s" \
      --director-style "BHASKAR_ABBURI_RAVI"

Outputs 3 files automatically:
1. `<movie_name>_scenes.csv`   -> Spreadsheet-ready CSV with Scene_ID, Start_Time, End_Time, Emotion_Tag,
                                 Genre_Tag, Dialect_Tag, English_CMI_Pct, Original_Telugu,
                                 P1_Standard_Tenglish, P2_Minimal_Tenglish, P4_Broken_Acrophonic_Tenglish.
2. `<movie_name>_scenes.txt`   -> Clean, human-readable scene-by-scene script with control tags (`[EMOTION: ...]`).
3. `<movie_name>_scenes.jsonl` -> Machine-ready JSONL for SLM training.
"""

import argparse
import asyncio
import csv
import json
import os
import re
import subprocess
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple



# ---------------------------------------------------------------------------
# 1. Automated Scene Emotion & Dialect Classifier (Lexical + Optional Gemini Flash API)
# ---------------------------------------------------------------------------

EMOTION_KEYWORDS: Dict[str, List[str]] = {
    "HEARTBREAK": [
        "బాధ", "ఏడుపు", "మోసం", "వదిలేసి", "మర్చిపోవడం", "మర్చిపోలేను", "చచ్చిపోతా",
        "ఒంటరి", "తట్టుకోలేను", "వద్దు", "నచ్చట్లేదు", "వెళ్ళిపోతున్నాను", "నాటకం",
        "కోరుకున్నది", "దొరక్కపోతే", "pain", "hurt", "cry", "breakup", "sorry",
    ],
    "ANGRY_WARNING": [
        "చంపేస్తా", "నరికేస్తా", "కత్తి", "రక్తం", "ఎదురు", "జాగ్రత్త", "వార్నింగ్",
        "కోపం", "తప్పు", "ఏంట్రా", "ఏందిరా", "మూసుకో", "కొడతా", "warning", "dare",
    ],
    "COMIC_TIMING": [
        "నవ్వు", "సరదా", "ఐస్", "క్రీమ్", "కిక్", "సెలవు", "పుట్టింటికి", "డబ్బా",
        "ఉప్పు", "కారం", "మెస్", "ఫైన్", "బొమ్మరిల్లు", "సంతకం", "లోన్", "కాఫీ",
        "funny", "joke", "ice", "cream", "kitchen", "mess",
    ],
    "SARCASTIC_ROAST": [
        "అంతేగా", "బిల్డప్", "హ్యాండ్", "సాకు", "మునిగిపోతవ్", "లైట్", "డెలులు",
        "ఘోస్ట్", "వైబ్", "ఆప్షన్", "ఇంకో నాన్న", "డిసైడ్", "షర్ట్", "షూ",
        "buildup", "ghost", "delulu", "vibe", "hand", "lite",
    ],
    "PHILOSOPHICAL": [
        "జీవితం", "మనిషికి", "మాటే", "ప్రాణం", "విలువ", "యుద్ధం", "యుద్ధంలో",
        "గెలవడం", "శత్రువు", "ఓడిపోయినా", "ధైర్యం", "నిజాయితీ", "బాధ్యత",
        "సొంతంగా", "నడవడం", "వీలైతే", "నాలుగు మాటలు",
    ],
    "ELEVATION_PUNCH": [
        "కమిట్", "భయం", "ఆపరేషన్", "హ్యాపీగా", "హ్యాపీనెస్", "లైఫ్ లాంగ్",
        "లెక్క", "చరిత్ర", "రికార్డ్", "పవర్", "commit", "power", "record",
    ],
}

DIALECT_KEYWORDS: Dict[str, List[str]] = {
    "TELANGANA": ["ఏంది", "భయ్యా", "అంటున్నవ్", "చెప్పిండు", "చేసిండు", "అయితది", "అయితంది", "దోస్తు", "బై", "గట్ల", "కూసున్నవ్"],
    "RAYALASEEMA": ["ఏందిరా", "పొలిమేర", "కత్తి", "రక్తం", "నరికేస్తా", "సామి", "అప్పా"],
    "COASTAL_ANDHRA": ["ఏమండోయ్", "గారూ", "అబ్బాయ్", "ఆవిడ", "కదండీ", "చెప్పమంటారు"],
}


def detect_scene_emotion_and_dialect(turns: List[str], cmi_pct: float) -> Tuple[str, str, str]:
    """
    Classifies a scene's [EMOTION], [GENRE], and [DIALECT] from its Telugu/Tenglish dialogue turns.
    """
    joined = " ".join(turns).lower()

    # 1. Score Emotions
    scores: Dict[str, int] = {emo: 0 for emo in EMOTION_KEYWORDS}
    for emo, kw_list in EMOTION_KEYWORDS.items():
        for kw in kw_list:
            if kw.lower() in joined:
                scores[emo] += 2

    best_emotion = max(scores.items(), key=lambda x: x[1])[0]
    if scores[best_emotion] == 0:
        best_emotion = "CONVERSATIONAL_DRAMA"

    # 2. Map Emotion -> Genre
    genre_map = {
        "HEARTBREAK": "FAMILY_DRAMA",
        "ANGRY_WARNING": "MASS_ACTION",
        "COMIC_TIMING": "SATIRE_COMEDY",
        "SARCASTIC_ROAST": "ROMCOM",
        "PHILOSOPHICAL": "FAMILY_DRAMA",
        "ELEVATION_PUNCH": "MASS_ACTION",
        "CONVERSATIONAL_DRAMA": "ROMCOM" if cmi_pct >= 12.0 else "FAMILY_DRAMA",
    }
    best_genre = genre_map.get(best_emotion, "FAMILY_DRAMA")

    # 3. Detect Dialect
    best_dialect = "URBAN_HYDERABAD" if cmi_pct >= 10.0 else "STANDARD_TELUGU"
    for dial, kw_list in DIALECT_KEYWORDS.items():
        if any(kw in joined for kw in kw_list):
            best_dialect = dial
            break

    return best_emotion, best_genre, best_dialect


def format_seconds_to_hms(seconds: float) -> str:
    """Formats seconds (e.g. 3725.4) into 'HH:MM:SS'."""
    s = int(max(0, seconds))
    hrs = s // 3600
    mins = (s % 3600) // 60
    secs = s % 60
    return f"{hrs:02d}:{mins:02d}:{secs:02d}"


# ---------------------------------------------------------------------------
# 2. Timestamped Scene Slicer (Preserves Start & End Timestamps per Scene)
# ---------------------------------------------------------------------------

def slice_movie_blocks_with_timestamps(
    blocks: List[SubtitleBlock],
    movie_name: str,
    era: str,
    director_style: str,
    same_turn_gap_sec: float = 1.2,
    new_scene_gap_sec: float = 4.5,
    max_turns_per_scene: int = 4,
) -> List[Dict[str, Any]]:
    """
    Slices an entire movie's subtitle blocks into timestamped 2-to-4 turn scenes,
    auto-tags Emotion/Genre/Dialect/CMI %, and generates all 4 Tenglish Personas.
    """
    if not blocks:
        return []

    raw_grouped_scenes: List[Tuple[float, float, List[str]]] = []
    cur_turns: List[str] = []
    cur_buf: List[str] = [blocks[0].text]
    scene_start = blocks[0].start_sec
    prev_end = blocks[0].end_sec

    for b in blocks[1:]:
        gap = b.start_sec - prev_end
        if gap <= same_turn_gap_sec:
            cur_buf.append(b.text)
        elif gap <= new_scene_gap_sec and len(cur_turns) < max_turns_per_scene - 1:
            cur_turns.append(" ".join(cur_buf))
            cur_buf = [b.text]
        else:
            cur_turns.append(" ".join(cur_buf))
            if len(cur_turns) >= 2:
                raw_grouped_scenes.append((scene_start, prev_end, cur_turns[:max_turns_per_scene]))
            cur_turns = []
            cur_buf = [b.text]
            scene_start = b.start_sec
        prev_end = b.end_sec

    if cur_buf:
        cur_turns.append(" ".join(cur_buf))
    if len(cur_turns) >= 2:
        raw_grouped_scenes.append((scene_start, prev_end, cur_turns[:max_turns_per_scene]))

    # Process each scene into rich tagged dictionary
    tagged_scenes: List[Dict[str, Any]] = []
    for idx, (st_sec, end_sec, turns) in enumerate(raw_grouped_scenes, start=1):
        joined_te = " ".join(turns)
        cmi_info = compute_utterance_cmi(joined_te)
        cmi_pct = cmi_info["en_lexeme_pct"]

        emotion, genre, dialect = detect_scene_emotion_and_dialect(turns, cmi_pct)

        p1_turns = [romanize_with_persona(t, "P1_STANDARD", "ERA_2025") for t in turns]
        p2_turns = [romanize_with_persona(t, "P2_MINIMAL", "ERA_2025") for t in turns]
        p3_turns = [romanize_with_persona(t, "P3_EXPRESS", "ERA_2025") for t in turns]
        p4_turns = [romanize_with_persona(t, "P4_ACROPHON", "ERA_2025") for t in turns]

        tagged_scenes.append(
            {
                "scene_id": f"{movie_name}_SC_{idx:04d}",
                "movie_name": movie_name,
                "start_time": format_seconds_to_hms(st_sec),
                "end_time": format_seconds_to_hms(end_sec),
                "era": era,
                "director_style": director_style,
                "emotion": emotion,
                "genre": genre,
                "dialect": dialect,
                "en_cmi_pct": cmi_pct,
                "en_word_count": cmi_info["counts"]["EN"],
                "hybrid_word_count": cmi_info["counts"]["HYBRID_FUSED"],
                "light_verb_count": cmi_info["counts"]["LIGHT_VERB"],
                "turn_count": len(turns),
                "turns_telugu": turns,
                "turns_p1_standard": p1_turns,
                "turns_p2_minimal": p2_turns,
                "turns_p3_expressive": p3_turns,
                "turns_p4_acrophonic": p4_turns,
            }
        )

    return tagged_scenes


# ---------------------------------------------------------------------------
# 3. Export to .CSV, .TXT (Scene-Based Script), and .JSONL
# ---------------------------------------------------------------------------

def export_movie_to_csv_txt_jsonl(
    scenes: List[Dict[str, Any]],
    movie_name: str,
    out_dir: str | Path,
) -> Dict[str, Path]:
    """
    Writes the extracted & tagged movie scenes to:
    1. `<movie_name>_scenes.csv`
    2. `<movie_name>_scenes.txt`
    3. `<movie_name>_scenes.jsonl`
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    csv_file = out_path / f"{movie_name}_scenes.csv"
    txt_file = out_path / f"{movie_name}_scenes.txt"
    jsonl_file = out_path / f"{movie_name}_scenes.jsonl"

    # 1. Write .CSV (Spreadsheet / Pandas / Google Sheets ready)
    csv_columns = [
        "scene_id",
        "movie_name",
        "start_time",
        "end_time",
        "era",
        "director_style",
        "emotion",
        "genre",
        "dialect",
        "en_cmi_pct",
        "turn_count",
        "dialogue_telugu",
        "dialogue_tenglish_standard",
        "dialogue_tenglish_minimal",
        "dialogue_tenglish_broken_acrophonic",
    ]
    with open(csv_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=csv_columns)
        writer.writeheader()
        for sc in scenes:
            writer.writerow(
                {
                    "scene_id": sc["scene_id"],
                    "movie_name": sc["movie_name"],
                    "start_time": sc["start_time"],
                    "end_time": sc["end_time"],
                    "era": sc["era"],
                    "director_style": sc["director_style"],
                    "emotion": sc["emotion"],
                    "genre": sc["genre"],
                    "dialect": sc["dialect"],
                    "en_cmi_pct": sc["en_cmi_pct"],
                    "turn_count": sc["turn_count"],
                    "dialogue_telugu": " // ".join(sc["turns_telugu"]),
                    "dialogue_tenglish_standard": " // ".join(sc["turns_p1_standard"]),
                    "dialogue_tenglish_minimal": " // ".join(sc["turns_p2_minimal"]),
                    "dialogue_tenglish_broken_acrophonic": " // ".join(sc["turns_p4_acrophonic"]),
                }
            )

    # 2. Write .TXT (Clean Scene-by-Scene Tagged Script)
    with open(txt_file, "w", encoding="utf-8") as f:
        f.write(f"================================================================================\n")
        f.write(f"MOVIE SCRIPT & TENGLISH CORPUS: {movie_name} ({len(scenes)} Scenes)\n")
        f.write(f"================================================================================\n\n")
        for sc in scenes:
            header = (
                f"[{sc['scene_id']}] [{sc['start_time']} -> {sc['end_time']}] "
                f"[ERA: {sc['era']}] [STYLE: {sc['director_style']}] "
                f"[EMOTION: {sc['emotion']}] [GENRE: {sc['genre']}] "
                f"[DIALECT: {sc['dialect']}] [EN_CMI: {sc['en_cmi_pct']}%]"
            )
            f.write(header + "\n")
            f.write("-" * len(header) + "\n")
            f.write("  [TELUGU SCRIPT]\n")
            for i, line in enumerate(sc["turns_telugu"], start=1):
                f.write(f"    Speaker {i}: {line}\n")
            f.write("  [STANDARD TENGLISH (P1)]\n")
            for i, line in enumerate(sc["turns_p1_standard"], start=1):
                f.write(f"    Speaker {i}: {line}\n")
            f.write("  [MINIMALIST TENGLISH (P2)]\n")
            for i, line in enumerate(sc["turns_p2_minimal"], start=1):
                f.write(f"    Speaker {i}: {line}\n")
            f.write("  [BROKEN / ACROPHONIC TENGLISH (P4)]\n")
            for i, line in enumerate(sc["turns_p4_acrophonic"], start=1):
                f.write(f"    Speaker {i}: {line}\n")
            f.write("\n")

    # 3. Write .JSONL
    with open(jsonl_file, "w", encoding="utf-8") as f:
        for sc in scenes:
            f.write(json.dumps(sc, ensure_ascii=False) + "\n")

    return {"csv": csv_file, "txt": txt_file, "jsonl": jsonl_file}


# ---------------------------------------------------------------------------
# 4. YouTube Link Downloader + Full CLI Entrypoint
# ---------------------------------------------------------------------------

def download_srt_from_youtube(youtube_url: str, work_dir: Path, movie_name: str) -> Optional[Path]:
    """
    Uses yt-dlp (and optional Whisper fallback) to extract the full movie .srt from a YouTube link.
    """
    work_dir.mkdir(parents=True, exist_ok=True)
    out_template = str(work_dir / f"{movie_name}.%(ext)s")
    cmd = [
        "yt-dlp",
        "--write-sub",
        "--write-auto-sub",
        "--sub-lang",
        "te,en",
        "--sub-format",
        "srt",
        "--convert-subs",
        "srt",
        "--skip-download",
        "-o",
        out_template,
        youtube_url,
    ]
    print(f"[YT-DLP] Extracting subtitles from YouTube URL: {youtube_url} ...")
    subprocess.run(cmd, check=False)
    srts = sorted(work_dir.glob(f"{movie_name}*.srt"))
    return srts[0] if srts else None


def extract_full_movie_pipeline(
    movie_name: str,
    era: str = "2000s",
    director_style: str = "BHASKAR_ABBURI_RAVI",
    youtube_url: Optional[str] = None,
    srt_file: Optional[str | Path] = None,
    out_dir: str | Path = "./drive_mirror/Tenglish_LM/extracted_movies",
) -> Dict[str, Any]:
    """
    Main function you call in Colab or Python:
    Give it a `youtube_url` (or `srt_file`) -> outputs `.csv`, `.txt`, and `.jsonl` with scene emotion tags.
    """
    out_path = Path(out_dir) / movie_name
    out_path.mkdir(parents=True, exist_ok=True)

    target_srt: Optional[Path] = Path(srt_file) if srt_file else None
    if youtube_url and (target_srt is None or not target_srt.exists()):
        target_srt = download_srt_from_youtube(youtube_url, out_path, movie_name)

    if target_srt is None or not target_srt.exists():
        raise FileNotFoundError(
            f"Could not find or download .srt for {movie_name}. "
            f"In Colab, ensure yt-dlp is installed (`!pip install -q yt-dlp`) or pass `--srt-file`."
        )

    srt_text = target_srt.read_text(encoding="utf-8", errors="replace")
    blocks = parse_srt_content(srt_text)
    scenes = slice_movie_blocks_with_timestamps(
        blocks=blocks,
        movie_name=movie_name,
        era=era,
        director_style=director_style,
    )
    exported = export_movie_to_csv_txt_jsonl(scenes, movie_name, out_path)

    return {
        "movie_name": movie_name,
        "subtitle_blocks_parsed": len(blocks),
        "scenes_extracted": len(scenes),
        "files": {k: str(v) for k, v in exported.items()},
        "emotion_distribution": {
            emo: sum(1 for s in scenes if s["emotion"] == emo)
            for emo in sorted({s["emotion"] for s in scenes})
        },
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Extract a full YouTube Telugu movie into Scene-Tagged .CSV and .TXT files.")
    parser.add_argument("--url", type=str, default=None, help="YouTube full movie URL")
    parser.add_argument("--srt-file", type=str, default=None, help="Local .srt subtitle file path")
    parser.add_argument("--movie-name", type=str, default="Bommarillu_2006", help="Movie identifier")
    parser.add_argument("--era", type=str, default="2000s", help="Movie era tag (1980s, 1990s, 2000s, 2010s, 2020s)")
    parser.add_argument("--director-style", type=str, default="BHASKAR_ABBURI_RAVI", help="Director/Writer style tag")
    parser.add_argument("--out-dir", type=str, default="./drive_mirror/Tenglish_LM/extracted_movies", help="Output folder")
    args = parser.parse_args()

    res = extract_full_movie_pipeline(
        movie_name=args.movie_name,
        era=args.era,
        director_style=args.director_style,
        youtube_url=args.url,
        srt_file=args.srt_file,
        out_dir=args.out_dir,
    )
    print(json.dumps(res, indent=2, ensure_ascii=False))


print('✅ Tenglish-Evo Engine & Scene Tagger loaded successfully!')


In [ ]:
# @title Step 3: Paste Any YouTube Telugu Trailer / Movie Scene URL & Run!
import pandas as pd
import subprocess

# 👇 PASTE ANY YOUTUBE TELUGU TRAILER OR SCENE URL HERE:
YOUTUBE_URL = "https://www.youtube.com/watch?v=e4N9al7vhVQ"  # @param {type:"string"}
TRAILER_NAME = "Telugu_Trailer_Test_01"  # @param {type:"string"}
ERA_TAG = "2020s"  # @param ["1980s", "1990s", "2000s", "2010s", "2020s"]
DIRECTOR_STYLE = "THARUN_TELANGANA_BANTER"  # @param ["TRIVIKRAM_ANTITHESIS", "PURI_CYNICAL_PUNCH", "THARUN_TELANGANA_BANTER", "BHASKAR_ABBURI_RAVI", "SUKUMAR_LOGIC", "VIVEK_URBAN_WIT", "JANDHYALA_WORDPLAY"]
USE_WHISPER_IF_NO_SUBS = True  # @param {type:"boolean"}

work_dir = OUT_DIR / TRAILER_NAME
work_dir.mkdir(parents=True, exist_ok=True)

# 1. Try downloading official/auto captions via yt-dlp
srt_path = download_srt_from_youtube(YOUTUBE_URL, work_dir, TRAILER_NAME)

# 2. If the trailer has NO captions on YouTube, optionally transcribe audio with Faster-Whisper!
if srt_path is None and USE_WHISPER_IF_NO_SUBS:
    print("⚠️ No caption track found on YouTube. Downloading audio & running Whisper ASR...")
    os.system("pip install -q faster-whisper")
    audio_path = work_dir / f"{TRAILER_NAME}.mp3"
    subprocess.run(["yt-dlp", "-x", "--audio-format", "mp3", "-o", str(audio_path), YOUTUBE_URL], check=False)
    if audio_path.exists():
        from faster_whisper import WhisperModel
        model = WhisperModel("small", device="cpu", compute_type="int8")
        segments, _ = model.transcribe(str(audio_path), language="te")
        srt_lines = []
        for idx, seg in enumerate(segments, start=1):
            st = format_seconds_to_hms(seg.start) + ",000"
            et = format_seconds_to_hms(seg.end) + ",000"
            srt_lines.append(f"{idx}\n{st} --> {et}\n{seg.text.strip()}\n")
        srt_path = work_dir / f"{TRAILER_NAME}.te.srt"
        srt_path.write_text("\n".join(srt_lines), encoding="utf-8")
        print("✅ Whisper ASR generated Telugu .srt:", srt_path)

# 3. Fallback demo if URL is geo-blocked or offline: run on built-in Bommarillu / Trailer SRT
if srt_path is None or not srt_path.exists():
    print("ℹ️ Using built-in Telugu Trailer + Bommarillu test SRT...")
    demo_srt = work_dir / f"{TRAILER_NAME}_demo.srt"
    demo_scenes = build_curated_multi_era_seed_corpus()
    demo_lines = []
    t_sec = 5
    idx = 1
    for sc in demo_scenes[:6]:
        for turn in sc.turns:
            st = format_seconds_to_hms(t_sec) + ",000"
            et = format_seconds_to_hms(t_sec + 3) + ",000"
            demo_lines.append(f"{idx}\n{st} --> {et}\n{turn}\n")
            t_sec += 4
            idx += 1
        t_sec += 6
    demo_srt.write_text("\n".join(demo_lines), encoding="utf-8")
    srt_path = demo_srt

# 4. Run the Scene Slicer, Emotion Auto-Tagger & 4-Persona Tenglish Generator
result = extract_full_movie_pipeline(
    movie_name=TRAILER_NAME,
    era=ERA_TAG,
    director_style=DIRECTOR_STYLE,
    srt_file=srt_path,
    out_dir=OUT_DIR,
)

print("\n==================================================================")
print("✅ EXTRACTION COMPLETE!")
print("==================================================================")
print(json.dumps(result, indent=2, ensure_ascii=False))

# 5. Preview the generated .CSV right inside Colab
df = pd.read_csv(result["files"]["csv"])
display(df[["scene_id", "start_time", "end_time", "emotion", "dialect", "en_cmi_pct", "dialogue_telugu", "dialogue_tenglish_standard", "dialogue_tenglish_broken_acrophonic"]])


In [ ]:
# @title Step 4: View the Generated Scene-by-Scene `.txt` Script
txt_path = Path(result["files"]["txt"])
print(txt_path.read_text(encoding="utf-8")[:4000])
